In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 1. Load the image and convert to grayscale ('L')
# Make sure "point_cloud.jpg" is in the same folder as your script
img = Image.open('logo.jpg').convert('L')
img_array = np.array(img)

# 2. Thresholding: The logo is dark, background is light.
# We create a "mask" where pixels darker than 128 are True (our logo)
foreground_mask = img_array < 128

# 3. Extract the row (y) and column (x) indices of the true pixels
y_coords, x_coords = np.where(foreground_mask)

# 4. Flip the Y-axis! 
# Image coordinates start with (0,0) at the TOP-left.
# Math coordinates start with (0,0) at the BOTTOM-left. 
y_coords = img_array.shape[0] - y_coords

# 5. Stack them into an (N, 2) array of [x, y] points
points = np.column_stack((x_coords, y_coords)).astype(np.float32)

# 6. Dequantization (The Magic Step)
# Add uniform noise between -0.5 and 0.5 to prevent the model from cheating
noise = np.random.uniform(-0.5, 0.5, size=points.shape)
points += noise

# 7. Subsampling (Optional but highly recommended for speed)
# A large image might have 100,000+ foreground pixels. Let's grab just 10,000 to train faster.
num_samples = 50000
if len(points) > num_samples:
    indices = np.random.choice(len(points), size=num_samples, replace=False)
    points = points[indices]

# 8. Normalization
# Center the data at (0,0)
points -= np.mean(points, axis=0)
# Scale it down so it fits nicely in a standard window (e.g., between -3 and 3)
max_val = np.max(np.abs(points))
X = (points / max_val) * 3.0

# Setup the variables just like the tutorial 
xlim, ylim = [-4, 4], [-4, 4]

# Plot to verify it worked
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], s=1, color='blue', alpha=0.5)
plt.xlim(xlim)
plt.ylim(ylim)
# Keeps the logo from stretching
plt.gca().set_aspect('equal', adjustable='box') 
plt.title("JMU Target Point Cloud")
plt.show()